# 02 Model: from baseline to a cost-based threshold

**Step 4:** split the data and set the baseline every model must beat.
**Step 5:** logistic regression, then a random forest.
**Step 6:** evaluate on the test set and choose the threshold from business costs.

In [1]:
import matplotlib.pyplot as plt

# One consistent, colour-blind-safe look for every chart in this project
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GREY, "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.titlelocation": "left", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "font.size": 10,
    "legend.frameon": False, "lines.linewidth": 2,
    "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW]),
})

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
clean = pd.read_csv("../data/processed/churn_clean.csv")
FEATURES = [c for c in clean.columns if c not in ("customer_id", "churn")]
X, y = clean[FEATURES], clean["churn"]

## 1. Train/test split

25% of customers are held back as a test set and not touched until Step 6. The split is **stratified** on churn so both sets have the same churn rate (about 15.6%). Without stratification a small test set can end up with noticeably more or fewer churners just by chance.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
ids_test = clean.loc[X_test.index, "customer_id"]
print(f"Train: {len(X_train)} customers, churn rate {y_train.mean():.1%}")
print(f"Test:  {len(X_test)} customers, churn rate {y_test.mean():.1%}")

Train: 2137 customers, churn rate 15.6%
Test:  713 customers, churn rate 15.7%


## 2. Baseline: predict 'no churn' for everyone

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred = baseline.predict(X_test)
print(f"Accuracy:  {accuracy_score(y_test, pred):.1%}")
print(f"Recall:    {recall_score(y_test, pred):.1%}  (churners caught)")
print(f"Precision: {precision_score(y_test, pred, zero_division=0):.1%}  (undefined: it flags nobody)")

Accuracy:  84.3%
Recall:    0.0%  (churners caught)
Precision: 0.0%  (undefined: it flags nobody)


The baseline is about 84% accurate and catches **zero** churners. This is the accuracy trap: on imbalanced data, a useless model can look good. Every model from here is judged on recall, precision and ranking quality, not accuracy.